Day 2 — Dataset & Preprocessing
================================
Topics covered:
  1.  Folder structure expected & how ImageFolder works
  2.  Why we normalise — pixel stats computed from scratch
  3.  Transform pipelines — train vs val/test difference
  4.  Custom Dataset class — __init__, __len__, __getitem__
  5.  Stratified train/val/test split — preserving class ratios
  6.  DataLoader — batching, shuffling, parallel workers
  7.  Batch visualisation — sanity check before training
  8.  Class imbalance analysis — what to do about it
  9.  Dataset statistics — mean & std computed from training set
  10. Full pipeline verification — shapes, types, ranges

In [113]:
import os
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from PIL import Image
from collections import Counter
 
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [114]:
# 1. EXPECTED FILE STRUCTURE
"""
Your dataset folder must look like this:
 
  data/
  ├── glioma/
  │     ├── image_001.jpg
  │     ├── image_002.jpg
  │     └── ...
  ├── meningioma/
  │     ├── image_001.jpg
  │     └── ...
  ├── notumour/
  │     └── ...
  └── pituitary/
        └── ...
 
torchvision.ImageFolder automatically:
  - Assigns class index by alphabetical folder order
  - glioma=0, meningioma=1, notumour=2, pituitary=3
  - Returns (PIL Image, label_int) for each sample
"""

#Creating Synthetic Data
print("Creating synthetic MRI-like dataset for demonstration...")
DATA_DIR = "data/brain_tumour"
CLASSES  = ["glioma", "meningioma", "notumour", "pituitary"]

CLASS_COUNTS = {"glioma":300, "meningioma":200, "pituitary":250, "notumour":250}

def make_synthetic_mri(class_name:str, seed:int)->np.ndarray:
    """
    Generate a synthetic 128x128 grayscale image that loosely
    resembles different tumour types — for demonstration only.
    Real Kaggle MRI images will replace these.
    """
    rng = np.random.default_rng(seed)
    base = rng.uniform(0.1,0.3,(128,128)).astype(np.float32)

    #Brain circular region
    y,x = np.ogrid[-64:64,-64:64]
    brain_mask = (x**2 + y**2)<55**2
    base[brain_mask] +=0.4

    if class_name == "glioma":
        # Irregular bright region (simulates infiltrative tumour)
        cy, cx = rng.integers(30, 90, 2)
        for _ in range(8):
            ry, rx = rng.integers(5, 8, 2)
            dy, dx = rng.integers(-10, 10, 2)
            mask = ((np.ogrid[-64:64, -64:64][0] - (cy-64+dy))**2 / ry**2 +
                    (np.ogrid[-64:64, -64:64][1] - (cx-64+dx))**2 / rx**2) < 1
            base[mask] += rng.uniform(0.5, 0.9)
    
    elif class_name == "meningioma":
        # Well-defined circular bright region near edge
        cy, cx = rng.integers(20, 40, 2)
        r = rng.integers(12, 22)
        mask = (y - (cy-64))**2 + (x - (cx-64))**2 < r**2
        base[mask] += 0.5

    elif class_name == "pituitary":
        # Small bright spot near centre-bottom
        cy, cx = 80, rng.integers(55, 75)
        r = rng.integers(5, 10)
        mask = (y - (cy-64))**2 + (x - (cx-64))**2 < r**2
        base[mask] += 0.6
    
    # notumour: just the brain base, no extra region
 
    base += rng.normal(0, 0.04, (128, 128)).astype(np.float32)
    return np.clip(base, 0, 1)

#Generate Synthetic images
for cls,count in CLASS_COUNTS.items():
    cls_dir = os.path.join(DATA_DIR,cls)
    os.makedirs(cls_dir,exist_ok=True)
    if len(os.listdir(cls_dir))<count:
        for i in range(count):
            img_array = make_synthetic_mri(cls, seed=i*100+hash(cls)%1000) #randomizer(not by me)
            img_pil = Image.fromarray((img_array*255).astype(np.uint8),mode='L')
            img_pil.save(os.path.join(cls_dir,f"{cls}_{i:04d}.png"))

total = sum(CLASS_COUNTS.values())
print(f"Synthetic dataset created: {total} images across {len(CLASSES)} classes")
for cls, count in CLASS_COUNTS.items():
    print(f"  {cls:15s}: {count:4d} images  ({count/total*100:.1f}%)")


Creating synthetic MRI-like dataset for demonstration...
Synthetic dataset created: 1000 images across 4 classes
  glioma         :  300 images  (30.0%)
  meningioma     :  200 images  (20.0%)
  pituitary      :  250 images  (25.0%)
  notumour       :  250 images  (25.0%)


In [115]:
# 2. PIXEL NORMALISATION
"""
Raw pixels: integers 0-255
Network weights: small floats ~0.01 (Kaiming init)
 
If inputs are 0-255 and weights are 0.01:
  weighted_sum = 0.01 * 200 + 0.01 * 150 + ... = huge number
  → activations explode on the first forward pass
 
Fix: normalise inputs to match weight scale
  Step 1: pixel / 255.0          → range [0, 1]
  Step 2: (pixel - mean) / std   → range approx [-2, 2]
 
Mean and std are computed from TRAINING SET ONLY.
Using test set stats = data leakage (cheating).
"""
# Compute mean and std from training images manually
print("Computing pixel mean and std from training images...")
raw_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128,128)),
    transforms.ToTensor(), # converts PIL [0,255] → tensor [0.0, 1.0]
])

#Load full dataset just to compute stats
full_ds_for_stats = ImageFolder(root=DATA_DIR, transform=raw_transform)

#training indices (first 70%) for stats
all_indices = list(range(len(full_ds_for_stats)))
all_labels  = [full_ds_for_stats.targets[i] for i in all_indices]

train_idx, temp_idx = train_test_split(all_indices,test_size=0.3,stratify=all_labels,random_state=42)

#Compute mean and std
pixel_sum = 0.0
pixel_sq_sum = 0.0 
n_pixels = 0

for idx in train_idx:
    img,_ = full_ds_for_stats[idx]
    pixel_sum += img.sum().item()
    pixel_sq_sum += (img**2).sum().item()
    n_pixels += img.numel()

TRAIN_MEAN = pixel_sum/n_pixels
TRAIN_STD = np.sqrt(pixel_sq_sum/n_pixels - TRAIN_MEAN**2)

print(f"  Training set mean: {TRAIN_MEAN:.4f}")
print(f"  Training set std:  {TRAIN_STD:.4f}")
print(f"  (These values will be used to normalise ALL splits)")

Computing pixel mean and std from training images...
  Training set mean: 0.4382
  Training set std:  0.2220
  (These values will be used to normalise ALL splits)


In [116]:
# 3. TRANSFORM PIPELINES

"""
WHY different transforms for train vs val/test?
 
Train:  We WANT variability → model sees flipped/rotated MRIs
        → learns features that are position/orientation invariant
        → reduces overfitting
 
Val/Test: We WANT consistency → same image always produces same output
          → metrics are reproducible and comparable across runs
          → no randomness in evaluation
"""
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    
    transforms.RandomHorizontalFlip(p=0.5), #p=0.5 means 50% chance of flipping each image
    transforms.RandomRotation(degrees=15), # Tumours can appear at any orientation
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Simulate different MRI scanner intensities
    transforms.ToTensor(), # PIL Image (H,W) → tensor (C,H,W), scales [0,255] → [0.0,1.0]
    transforms.Normalize(mean=[TRAIN_MEAN], std=[TRAIN_STD]), # (pixel - mean) / std → zero-centred, unit variance
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),

    transforms.ToTensor(),
    transforms.Normalize(mean=[TRAIN_MEAN], std=[TRAIN_STD]), # Same normalisation stats as training (not recomputed)
    #NO augmentations
])

print("Train transform pipeline:")
for t in train_transform.transforms:
    print(f"  → {t.__class__.__name__}")

print("\nVal/Test transform pipeline:")
for t in val_test_transform.transforms:
    print(f"  → {t.__class__.__name__}")

Train transform pipeline:
  → Grayscale
  → Resize
  → RandomHorizontalFlip
  → RandomRotation
  → ColorJitter
  → ToTensor
  → Normalize

Val/Test transform pipeline:
  → Grayscale
  → Resize
  → ToTensor
  → Normalize


In [117]:
# 4. CUSTOM DATASET CLASS
"""
Why write a custom Dataset instead of just using ImageFolder?
 
ImageFolder is convenient but inflexible:
  - Can't assign different transforms to specific indices
  - Can't easily do stratified splits
  - Can't add custom metadata (patient ID, scanner type, etc.)
 
Our custom Dataset wraps ImageFolder + accepts a subset of indices
+ applies the correct transform per split. This pattern scales to
any medical imaging task you'll encounter.
"""

class BrainTumourDataset(Dataset):
    """
    Custom Dataset for Brain Tumour MRI Classification.
 
    Wraps torchvision.ImageFolder but allows:
      - Subset selection (for train/val/test splits)
      - Per-split transform assignment
      - Easy access to class names and label mapping
 
    Args:
        root_dir  : path to dataset folder (contains class subfolders)
        indices   : list of indices into the full ImageFolder dataset
        transform : torchvision transform pipeline to apply
    """

    def __init__(self, root_dir:str, indices:list,transform = None):
        # ImageFolder scans root_dir, assigns labels alphabetically
        # class_to_idx: {'glioma':0, 'meningioma':1, 'notumour':2, 'pituitary':3}
        self.base_dataset = ImageFolder(root = root_dir)
        self.indices = indices
        self.transform = transform
        self.class_to_idx = self.base_dataset.class_to_idx
        self.idx_to_class = {v: k for k,v in self.class_to_idx.items()}
        self.classes = self.base_dataset.classes

    def __len__(self) -> int:
        #Called by DataLoader to know number of samples
        return len(self.indices)
    
    def __getitem__(self, i:int):
        """
        Called by DataLoader for each sample in a batch.
 
        i         : position within THIS split (0 to len-1)
        real_idx  : actual index in the full ImageFolder
        img       : PIL Image loaded from disk
        label     : integer class index (0,1,2,3)
 
        Returns: (tensor of shape (1,128,128), int label)
        """
        real_idx = self.indices[i]
        img,label = self.base_dataset[real_idx]
        if self.transform:
            img = self.transform(img)

        return img,label
    
    def get_class_name(self,label:int)->str:
        return self.idx_to_class[label]
    
    def get_class_distribution(self)->dict:
        labels = [self.base_dataset.targets[i] for i in self.indices]
        return {self.idx_to_class[cls]: count for cls,count in sorted(Counter(labels).items())}

print("BrainTumourDataset class defined.")
print("Key design decisions:")
print("  - Stores indices, not images → memory efficient")
print("  - PIL loading in __getitem__ → lazy, loads only when needed")
print("  - Transform applied AFTER loading → augmentation is per-call (random)")



BrainTumourDataset class defined.
Key design decisions:
  - Stores indices, not images → memory efficient
  - PIL loading in __getitem__ → lazy, loads only when needed
  - Transform applied AFTER loading → augmentation is per-call (random)


In [118]:
# 5. STRATIFIED TRAIN/VAL/TEST SPLIT
"""
Why stratified?
  Random split might give val/test sets with very few examples
  of a rare class (e.g. pituitary). Stratified split ensures
  every class appears proportionally in EVERY split.
 
Why 70/15/15?
  70% train  — model needs enough data to learn from
  15% val    — enough to get stable metric estimates during training
  15% test   — held out completely, only touched at the very end
 
IMPORTANT: test set is NEVER used during training or
hyperparameter tuning. It simulates unseen clinical data.
"""
# Full dataset (no transform — we'll assign transforms per split)
full_dataset = ImageFolder(root=DATA_DIR)
all_indices  = list(range(len(full_dataset)))
all_labels   = full_dataset.targets   # list of int labels

train_val_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.15,
    stratify=all_labels,
    random_state=42
)
# val is 15/85 ≈ 0.176 of the train_val portion
train_val_labels = [all_labels[i] for i in train_val_idx]
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.176,
    stratify=train_val_labels,
    random_state=42
)

print(f"Total samples:      {len(all_indices)}")
print(f"Train samples:      {len(train_idx)}  ({len(train_idx)/len(all_indices)*100:.1f}%)")
print(f"Val samples:        {len(val_idx)}   ({len(val_idx)/len(all_indices)*100:.1f}%)")
print(f"Test samples:       {len(test_idx)}  ({len(test_idx)/len(all_indices)*100:.1f}%)")

print("\nClass distribution verification:")
cls_names = full_dataset.classes
print(f"{'Class':<15} {'Full':>6} {'Train':>6} {'Val':>6} {'Test':>6}")
print("-" * 45)
 
for cls_idx, cls_name in enumerate(cls_names):
    full_c  = sum(1 for l in all_labels if l == cls_idx)
    train_c = sum(1 for i in train_idx  if all_labels[i] == cls_idx)
    val_c   = sum(1 for i in val_idx    if all_labels[i] == cls_idx)
    test_c  = sum(1 for i in test_idx   if all_labels[i] == cls_idx)
    print(f"{cls_name:<15} {full_c:>6} {train_c:>6} {val_c:>6} {test_c:>6}")

train_dataset = BrainTumourDataset(DATA_DIR, train_idx, transform=train_transform)
val_dataset   = BrainTumourDataset(DATA_DIR, val_idx,   transform=val_test_transform)
test_dataset  = BrainTumourDataset(DATA_DIR, test_idx,  transform=val_test_transform)
 
print(f"\nClass to index mapping: {train_dataset.class_to_idx}")

Total samples:      1000
Train samples:      700  (70.0%)
Val samples:        150   (15.0%)
Test samples:       150  (15.0%)

Class distribution verification:
Class             Full  Train    Val   Test
---------------------------------------------
glioma             300    210     45     45
meningioma         200    140     30     30
notumour           250    175     37     38
pituitary          250    175     38     37

Class to index mapping: {'glioma': 0, 'meningioma': 1, 'notumour': 2, 'pituitary': 3}


In [119]:
# 6. DATALOADERS
"""
DataLoader wraps a Dataset and handles:
 
  batch_size : how many samples per forward pass
               - Too small (1-4): noisy gradients, slow
               - Too large (256+): smooth gradients but needs more RAM
               - 32 is a reliable default for medical imaging
 
  shuffle    : True for train (random order prevents the model
               from learning the order of samples)
               False for val/test (deterministic evaluation)
 
  num_workers: parallel CPU processes for loading images
               - 0 = main process only (safe but slow)
               - 2-4 = parallel loading (faster)
               - Set to 0 on Windows to avoid multiprocessing issues
 
  pin_memory : True if using GPU — pre-loads batches to page-locked
               memory for faster CPU→GPU transfer
"""
BATCH_SIZE = 32
NUM_WORKERS = 0 #set 2-4 to GPU

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers=NUM_WORKERS,
    pin_memory = False,
    drop_last = True, # drop final incomplete batch
    # drop_last=True ensures every batch is exactly batch_size
    # important for BatchNorm which behaves oddly on batch_size=1
)

val_loader = DataLoader(
    val_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False, # ← deterministic for evaluation
    num_workers= NUM_WORKERS,
    pin_memory = False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
)

print(f"Train loader:  {len(train_loader)} batches x {BATCH_SIZE} = ~{len(train_loader)*BATCH_SIZE} samples ")
print(f"Val loader:   {len(val_loader)} batches")
print(f"Test loader:  {len(test_loader)} batches")

Train loader:  21 batches x 32 = ~672 samples 
Val loader:   5 batches
Test loader:  5 batches


In [120]:
# 7. BATCH VISUALISATION — SANITY CHECK
"""
Before training ANYTHING, always visualise a batch.
This catches:
  - Wrong normalisation (images look all-black or all-white)
  - Wrong channel order (grayscale loaded as RGB accidentally)
  - Label mismatch (image says 'glioma' but label says 3)
  - Wrong image size
"""

images,labels = next(iter(train_loader)) #one batch from train_loader
print(f"Batch shape:  {tuple(images.shape)}")
print(f"  N={images.shape[0]}, C={images.shape[1]}, H={images.shape[2]}, W={images.shape[3]}")
print(f"Labels shape: {tuple(labels.shape)}")
print(f"Label values: {labels[:8].tolist()}")
print(f"Pixel range after normalise: [{images.min():.3f}, {images.max():.3f}]")
print(f"Pixel mean:   {images.mean():.4f}  (should be ≈ 0)")
print(f"Pixel std:    {images.std():.4f}   (should be ≈ 1)")

# Denormalise for display
def denormalizer(tensor,mean,std):
    return tensor*std + mean

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.patch.set_facecolor('#F8F8F6')
fig.suptitle("Training batch — 32 samples (normalised then denormalised for display)",
             fontsize=11, fontweight='bold')
colours = {'glioma':'#E74C3C', 'meningioma':'#3498DB',
           'notumour':'#2ECC71', 'pituitary':'#9B59B6'}
for idx, ax in enumerate(axes.flatten()):
    if idx >= len(images): ax.axis('off'); continue
    img = denormalizer(images[idx].squeeze(), TRAIN_MEAN, TRAIN_STD)
    img = img.clamp(0, 1).numpy()
    lbl = labels[idx].item()
    cls = train_dataset.idx_to_class[lbl]
    ax.imshow(img, cmap='gray')
    ax.set_title(cls, fontsize=7, color=colours[cls], fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig("outputs/batch_visualisation.png", dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
print("Saved: outputs/batch_visualisation.png")

Batch shape:  (32, 1, 128, 128)
  N=32, C=1, H=128, W=128
Labels shape: (32,)
Label values: [2, 0, 0, 0, 1, 3, 2, 0]
Pixel range after normalise: [-1.974, 2.530]
Pixel mean:   -0.0276  (should be ≈ 0)
Pixel std:    1.0632   (should be ≈ 1)
Saved: outputs/batch_visualisation.png


In [125]:
# 8. CLASS IMBALANCE ANALYSIS
"""
Class imbalance = some classes have far more examples than others.
 
Why it's dangerous:
  If 70% of images are 'glioma', a model that ALWAYS predicts
  'glioma' gets 70% accuracy — while being completely useless.
  Accuracy alone is a misleading metric on imbalanced data.
 
How to handle it (we'll implement this properly on Day 9):
  Option 1: WeightedRandomSampler — oversample rare classes in batches
  Option 2: Class weights in loss  — penalise wrong predictions on
            rare classes more heavily
  Option 3: Collect more data      — ideal but not always possible
"""
train_labels = [train_dataset.base_dataset.targets[i] for i in train_idx]
class_counts = Counter(train_labels)

print("Training set class distribution:")
max_count = max(class_counts.values())
for cls_idx in sorted(class_counts):
    cls_name = train_dataset.idx_to_class[cls_idx]
    count    = class_counts[cls_idx]
    bar      = "█" * int(count / max_count * 30)
    print(f"  {cls_name:<15} {count:4d}  {bar}")

# Compute class weights for weighted loss (inverse frequency)
total_train = len(train_idx)
class_weights = torch.tensor([
    total_train/(len(class_counts)*class_counts[i])
    for i in range(len(class_counts))
],dtype = torch.float32)

print(f"\nClass weights for WeightedLoss (inverse frequency):")
for i, w in enumerate(class_weights):
    print(f"  {train_dataset.idx_to_class[i]:<15}: weight = {w:.3f}")
print("  (higher weight = model penalised more for missing that class)")

#Class distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#F8F8F6')

cls_names_list = [train_dataset.idx_to_class[i] for i in range(len(CLASSES))]
split_names    = ['Train', 'Val', 'Test']
split_indices  = [train_idx, val_idx, test_idx]
bar_colors     = ['#378ADD', '#1D9E75', '#D85A30']

counts_per_split = []
for split_idx in split_indices:
    split_labels = [all_labels[i] for i in split_idx]
    counts_per_split.append([
        sum(1 for l in split_labels if l == ci) for ci in range(len(CLASSES))
    ])
 
x = np.arange(len(CLASSES))
width = 0.25
for i, (name, counts, color) in enumerate(zip(split_names, counts_per_split, bar_colors)):
    axes[0].bar(x + i*width, counts, width, label=name, color=color, alpha=0.85)
 
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(cls_names_list, rotation=15, ha='right')
axes[0].set_title("Class distribution across splits", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Sample count")
axes[0].legend()

# Pie chart of full dataset
full_counts = [sum(1 for l in all_labels if l == i) for i in range(len(CLASSES))]
pie_colors  = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
axes[1].pie(full_counts, labels=cls_names_list, autopct='%1.1f%%',
            colors=pie_colors, startangle=90)
axes[1].set_title("Full dataset class proportions", fontsize=11, fontweight='bold')
 
plt.tight_layout()
plt.savefig("outputs/class_distribution.png", dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())

# Pie chart of full dataset
full_counts = [sum(1 for l in all_labels if l == i) for i in range(len(CLASSES))]
pie_colors  = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
axes[1].pie(full_counts, labels=cls_names_list, autopct='%1.1f%%',
            colors=pie_colors, startangle=90)
axes[1].set_title("Full dataset class proportions", fontsize=11, fontweight='bold')
 
plt.tight_layout()
plt.savefig("outputs/class_distribution.png", dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())

Training set class distribution:
  glioma           210  ██████████████████████████████
  meningioma       140  ████████████████████
  notumour         175  █████████████████████████
  pituitary        175  █████████████████████████

Class weights for WeightedLoss (inverse frequency):
  glioma         : weight = 0.833
  meningioma     : weight = 1.250
  notumour       : weight = 1.000
  pituitary      : weight = 1.000
  (higher weight = model penalised more for missing that class)


In [128]:
# 9. AUGMENTATION VISUALISATION

"""
Because augmentation is random, the SAME image produces
a DIFFERENT tensor every time __getitem__ is called.
This is intentional — it's what creates training variety.
"""
#One Image, Eight Transforms
sample_img_pil, sample_label = ImageFolder(root=DATA_DIR)[0]
sample_cls = cls_names[sample_label]

# Augment-only transform (no normalise, for display)
aug_only = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.patch.set_facecolor('#F8F8F6')
fig.suptitle(f"Same '{sample_cls}' image — 8 different augmentations",
             fontsize=11, fontweight='bold')

for ax in axes.flatten():
    augmented = aug_only(sample_img_pil).squeeze().numpy()
    ax.imshow(augmented, cmap='gray')
    ax.axis('off')
 
plt.tight_layout()
plt.savefig("outputs/augmentation_variety.png", dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())

C:\Users\AMAY M NAIR\AppData\Local\Temp\ipykernel_13124\3683461969.py:22: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(2, 4, figsize=(14, 7))


In [137]:
# 10. FULL PIPELINE VERIFICATION

checks = []

#Check1 : Batch Shape
imgs, lbls = next(iter(train_loader))
shape_ok = imgs.shape == torch.Size([BATCH_SIZE, 1, 128, 128])
checks.append(("Batch shape (32,1,128,128)", shape_ok))

#Check2 : Pixel range after normalize
range_ok = imgs.min().item() < -0.5 and imgs.max().item() > 0.5
checks.append(("Pixel normalized (not [0,1] anymore)",range_ok))

#Check3 : mean approc 0 after normalize
mean_ok = abs(imgs.mean().item()) < 0.3
checks.append(("Batch mean ≈ 0", mean_ok))

#Check4 : labels in range
labels_ok = lbls.min().item() >= 0 and lbls.max().item() <= 3
checks.append(("Labels in range [0,3]", labels_ok))

#Check5 : no NaN values
no_nan = not torch.isnan(imgs).any().item()
checks.append(("No NaN values in batch", no_nan))
 
#Check6 : all 4 classes appear in training set
train_class_set = set(all_labels[i] for i in train_idx)
all_classes_ok  = len(train_class_set) == 4
checks.append(("All 4 classes in training set", all_classes_ok))

#Check7 : no index overlap between splits
overlap_tv = set(train_idx) & set(val_idx)
overlap_tt = set(train_idx) & set(test_idx)
overlap_vt = set(val_idx)   & set(test_idx)
no_overlap = len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0
checks.append(("No overlap between train/val/test", no_overlap))

#Check8 : val/test transforms are deterministic
img1, _ = val_dataset[0]
img2, _ = val_dataset[0]
deterministic = torch.allclose(img1, img2)
checks.append(("Val/test transforms are deterministic", deterministic))

#Check9 : train transforms are random (different each call)
img1_t, _ = train_dataset[0]
img2_t, _ = train_dataset[0]
random_aug = not torch.allclose(img1_t, img2_t)
checks.append(("Train transforms are random (augmentation)", random_aug))

all_passed = True
for desc, result in checks:
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"  {desc:<43} {status:>8}")
    if not result:
        all_passed = False

print(f"  {'All checks passed' if all_passed else 'SOME CHECKS FAILED — fix before Day 3'}")


  Batch shape (32,1,128,128)                    ✓ PASS
  Pixel normalized (not [0,1] anymore)          ✓ PASS
  Batch mean ≈ 0                                ✓ PASS
  Labels in range [0,3]                         ✓ PASS
  No NaN values in batch                        ✓ PASS
  All 4 classes in training set                 ✓ PASS
  No overlap between train/val/test             ✓ PASS
  Val/test transforms are deterministic         ✓ PASS
  Train transforms are random (augmentation)    ✓ PASS
  All checks passed



  **Dataset size:    {len(all_indices)} images  |  4 classes**

  Train/Val/Test:  {len(train_idx)} / {len(val_idx)} / {len(test_idx)}

  Image size:      128 × 128 (grayscale, 1 channel)

  Batch size:      {BATCH_SIZE}

  Train batches:   {len(train_loader)}  per epoch
  
 
Normalisation:   mean={TRAIN_MEAN:.4f}, std={TRAIN_STD:.4f}

  Augmentations:   HFlip(p=0.5), Rotation(±15°), ColorJitter
 

  Class weights:   {[round(w.item(),3) for w in class_weights]}

  (use with nn.CrossEntropyLoss(weight=class_weights) on Day 6)
 
  Output files:
  
    outputs/batch_visualisation.png   ← 32 training samples

    outputs/class_distribution.png    ← imbalance analysis
    
    outputs/augmentation_variety.png  ← same image, 8 augmentations
